In [ ]:
import deepfmkit.core as dfm
from deepfmkit.plotting import default_rc
import matplotlib.pyplot as plt

plt.rcParams.update(default_rc)

dff = dfm.DeepFrame()

laser_config = dfm.LaserConfig(label="main_laser")
# Modulation frequency (Hz)
laser_config.fm = 1000
# Laser frequency noise at 1 Hz (Hz/rtHz)
laser_config.f_n = 1e6
# Laser relative intensity noise (1/rtHz)
laser_config.r_n = 1e-5

main_ifo_config = dfm.IfoConfig(label="dynamic_ifo")
# Reference arm length (m)
main_ifo_config.ref_arml = 0.1
# Measurement arm length (m)
main_ifo_config.meas_arml = 0.15
# Measurement arm modulation frequency (Hz)
main_ifo_config.arml_mod_f = 1.0
# Armlength modulation amplitude (m)
main_ifo_config.arml_mod_amp = 1e-9
# Armlength noise (m/rtHz)
main_ifo_config.arml_n = 1e-12

# Target effective modulation index (rad)
m_target = 6.0
laser_config.set_df_for_effect(main_ifo_config, m_target)

main_label = "dynamic_channel"
main_channel = dfm.SimConfig(
    label=main_label,
    laser_config=laser_config,
    ifo_config=main_ifo_config,
    f_samp=int(200e3),  # Sampling frequency (Hz)
)
dff.sims[main_label] = main_channel

ref_label = "reference_channel"
dff.create_witness_channel(
    main_channel_label=main_label,
    witness_channel_label=ref_label,
    m_witness=4.3,  # We want a witness with this effective modulation index
)

dff.simulate(
    label=main_label,
    witness_label=ref_label,
    n_seconds=10,
    verbose=True,
)

dff.sims[main_label].info()
dff.sims[ref_label].info()

for i, key in enumerate(dff.raws):
    print(f"Fitting channel '{key}'...")
    dff.fit(key, fit_label=f"fit_{key}")

ax = dff.plot()
plt.show()